# PyTorch CNN-LSTM Training

Train three-class PyTorch video classifiers from processed memmaps, with grouped video splits, two-step video aggregation, and final checkpoints. The independent efficiency benchmark at the end compares model inference cost without rerunning training.


In [ ]:
# Run this once if your notebook environment is missing PyTorch or training dependencies:
# %pip install torch torchvision scikit-learn opencv-python matplotlib h5py

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
BASE_DIR = next(
    (directory for directory in (start, *start.parents)
     if (directory / "src" / "cnn_lstm_pytorch.py").is_file()),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError("Start the notebook kernel inside the violence-movies project.")
SRC_DIR = BASE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cnn_lstm_pytorch import (
    CLASS_NAMES,
    combine_mmap_datasets,
    infer_mmap_chunk_count,
    get_device,
    get3DCNNLSTMModel,
    getC3DCNNModel,
    getLSTMModel,
    plot_final_training_history,
    plot_validation_curves,
    run_end_to_end_experiment,
    train_final_model,
    validate_mmap_dataset,
)

from video_models import getMViTv2SModel
from movinet_models import getMoViNetA0BaseModel

print(f"Base directory: {BASE_DIR}")
print(f"Device: {get_device()}")


## 1. Prepare Combined Dataset or Selected Dataset

### Combined Dataset Experiment (Choose one and skip the other)
This creates and validates `data/processed/combined-violence-datasets` from the AIRTLab-style `violence-detection-dataset` memmaps and the `XD-Violence` memmaps. XD video IDs are offset during the copy so grouped splits cannot mix chunks from different source videos.


In [ ]:
# AIRT_DATASET = BASE_DIR / "data" / "processed" / "violence-detection-dataset"
# XD_DATASET = BASE_DIR / "data" / "processed" / "XD-Violence"
# COMBINED_DATASET = BASE_DIR / "data" / "processed" / "combined-violence-datasets"

# # Set to True only when you intentionally want to recreate the combined memmaps.
# REBUILD_COMBINED_MMAPS = False

# combined_summary = combine_mmap_datasets(
#     [AIRT_DATASET, XD_DATASET],
#     COMBINED_DATASET,
#     rebuild=REBUILD_COMBINED_MMAPS,
# )

# OUT = COMBINED_DATASET
# print(f"Training dataset: {OUT}")
# print(f"Training chunks: {infer_mmap_chunk_count(OUT)}")


### AIRTLab-only experiment (Choose one and skip the other)

In [ ]:
OUT = BASE_DIR / "data" / "processed" / "violence-detection-dataset"

validate_mmap_dataset(OUT)
print(f"Training dataset: {OUT}")
print(f"Training chunks: {infer_mmap_chunk_count(OUT)}")

## 2. Select Model

MViTv2-S uses the **same existing 112x112 chunks** and source-file splits. Its wrapper converts BGR to RGB, scales pixels, directly upsamples to 224x224 without cropping, and normalizes. Both fine-tuning and evaluation use this adapter; no raw-video decoding or new prepared dataset is needed. Existing model defaults are unchanged.

MoViNet-A0-Base is an optional independent-chunk baseline. It uses the same 16-frame 112x112 BGR source chunks, with an internal RGB 0..1 adapter and full-frame upsampling to 172x172. It retains chunk cross-entropy and the existing two-step video decision. Native BatchNorm running statistics are fixed by default while backbone/classifier parameters fine-tune. Select `movinet_a0_base`; no dataset rebuild is required.


In [ ]:
from functools import partial
import torch

# If C3D weight conversion fails because h5py is missing, run this once:
# %pip install h5py

OUT = globals().get("OUT", BASE_DIR / "data" / "processed" / "combined-violence-datasets")

# Options: "3dcnnlstm", "c3dcnn", "lstm", "mvit_v2_s", "movinet_a0_base"
MODEL_NAME = "lstm"

MODEL_REGISTRY = {
    "movinet_a0_base": {
        "factory": getMoViNetA0BaseModel,
        "batch_size": 8,
        "checkpoint": BASE_DIR / "weights" / "MoViNetA0Base.pt",
        "model_kwargs": {"weights": "KINETICS600", "freeze_backbone": False, "freeze_bn_stats": True},
        "lr": 1e-5,
    },
    "3dcnnlstm": {
        "factory": get3DCNNLSTMModel,
        "batch_size": 64,
        "checkpoint": BASE_DIR / "weights" / "3DCNNLSTM.pt",
        # "lr": 1e-4,
    },
    "c3dcnn": {
        "factory": getC3DCNNModel,
        "batch_size": 16,
        "checkpoint": BASE_DIR / "weights" / "C3DCNN.pt",
    },
    "lstm": {
        "factory": getLSTMModel,
        "batch_size": 8,
        "checkpoint": BASE_DIR / "weights" / "LSTM.pt",
    },
    "mvit_v2_s": {
        "factory": getMViTv2SModel,
        "batch_size": 8,  # FP32 full fine-tuning; increase after checking GPU memory.
        "checkpoint": BASE_DIR / "weights" / "MViTv2S.pt",
        "model_kwargs": {"weights": "KINETICS400_V1", "freeze_backbone": False},
        "optimizer_factory": partial(torch.optim.AdamW, weight_decay=0.01),
        "lr": 1e-5,  # Starting value; select tuning settings using validation only.
    },
}

config = MODEL_REGISTRY[MODEL_NAME]
get_model = config["factory"]
BATCH_SIZE = config["batch_size"]
CHECKPOINT_PATH = config["checkpoint"]
MODEL_KWARGS = config.get("model_kwargs")
OPTIMIZER_FACTORY = config.get("optimizer_factory")
LEARNING_RATE = config.get("lr", 1e-4)

print(f"Dataset: {OUT}")
print(f"Model: {MODEL_NAME}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

print(f"Learning rate: {LEARNING_RATE}; model overrides: {MODEL_KWARGS}")
if MODEL_NAME == "mvit_v2_s":
    print("Same memmaps: BGR 16x112x112 -> internal RGB/resize/normalize -> 16x224x224")
    print("Kinetics-400 weights download on first construction; no dataset rebuild.")


## 3. Cross-Validation


In [ ]:
all_chunk_val_curves, all_video_val_curves = run_end_to_end_experiment(
    get_model=get_model,
    batch_size=BATCH_SIZE,
    dataset_base_path=OUT,
    mmap_dataset_base_path=OUT,
    samples_mmap_name="samples.mmap",
    labels_mmap_name="labels.mmap",
    end_to_end_model_name=MODEL_NAME,
    r_state=42,
    class_names=CLASS_NAMES,
    epochs=50,
    patience=10,
    lr=LEARNING_RATE,
    n_splits=10,
    num_workers=0,
    model_kwargs=MODEL_KWARGS,
    optimizer_factory=OPTIMIZER_FACTORY,
)

In [ ]:
plot_validation_curves(all_chunk_val_curves, all_video_val_curves)

## 4. Train One Final Model


In [ ]:
model, history = train_final_model(
    get_model=get_model,
    dataset_base_path=OUT,
    checkpoint_path=CHECKPOINT_PATH,
    batch_size=BATCH_SIZE,
    epochs=100,
    patience=10,
    lr=LEARNING_RATE,
    validation_size=0.1,
    random_state=42,
    num_workers=0,
    model_kwargs=MODEL_KWARGS,
    optimizer_factory=OPTIMIZER_FACTORY,
)

In [ ]:
plot_final_training_history(history)

## 5. Independent PyTorch efficiency benchmark

Restart the kernel and run this section to benchmark the saved checkpoints. Each worker loads weights on CPU, accepts the omitted C3D layer keys in legacy checkpoints, and transfers the constructed model to the selected device.

The supplementary table shows active and trainable parameters. Active parameters belong to layers used by the forward pass, including frozen layers. Trainable parameters have `requires_grad=True` under the benchmark model configuration. The detailed table also shows registered parameters. The C3D backbones stop at pool3 for the hybrid and fc6 for standalone C3D, so the active and registered counts agree for these cases. Counts exclude buffers. Runtime measures a device-ready forward call under `eval()` and `inference_mode()`, excluding loading, external preprocessing, input transfer, and state reset.

Missing checkpoints produce explicit missing rows. Set `allow_untrained=True` only for explicitly labelled architecture-only profiling. All four models use the same operator counter. It counts dense multiply-accumulates from convolutions, linear layers and matrix contractions, including attention scores, weighted values and relative position terms. The JSON report retains the operator counts and raw timings.

MViTv2-S is an optional fourth case (missing until its project checkpoint is trained). It receives the same BGR 16x112x112 source batch; its internal 224x224 adaptation is included in forward latency and GPU memory. The table identifies source/network resolutions. Its total includes contractions at the transformer’s internal resolution. Bias, elementwise arithmetic, activations, normalization, softmax, pooling and interpolation are excluded from the MAC convention.

MoViNet-A0-Base loads its separate project checkpoint and measures the same 16-frame source workload with its internal 112-to-172 adapter. This benchmark does not download Kinetics weights. The new model has a dedicated native convolution MAC counter; other cases retain their existing counters.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys

start = Path.cwd().resolve()
BENCHMARK_ROOT = next(
    (directory for directory in (start, *start.parents)
     if (directory / "src" / "cnn_lstm_pytorch.py").is_file()),
    None,
)
if BENCHMARK_ROOT is None:
    raise FileNotFoundError("Start the notebook kernel inside the violence-movies project.")
if str(BENCHMARK_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(BENCHMARK_ROOT / "src"))

from model_benchmark import (
    BenchmarkCase, BenchmarkConfig, Factory, MAC_SCOPE, cnn_benchmark_cases,
    run_benchmarks, results_to_dataframe, export_benchmark_report,
)

from video_models import mvit_benchmark_case
from movinet_models import movinet_base_benchmark_case


In [ ]:
BENCHMARK_BATCH_SIZE = 1  # Separate from every model's training batch size.
BENCHMARK_MODELS = ["c3dcnn", "lstm", "3dcnnlstm", "mvit_v2_s", "movinet_a0_base"]
BENCHMARK_CONFIG = BenchmarkConfig(
    device="auto",       # Or "cuda:0" / "cpu".
    dtype="float32",
    warmup=30,
    iterations=200,
    trials=3,
    seed=42,
    cpu_threads=1,
    allow_untrained=False, 
    allow_tf32=False,
)
SAVE_BENCHMARK_RESULTS = False
BENCHMARK_OUTPUT_DIR = BENCHMARK_ROOT / "results" / "benchmarks" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


In [ ]:
benchmark_cases = []
for name in BENCHMARK_MODELS:
    if name == "movinet_a0_base":
        benchmark_cases.append(movinet_base_benchmark_case(
            BENCHMARK_ROOT, batch_size=BENCHMARK_BATCH_SIZE,
        ))
    elif name == "mvit_v2_s":
        benchmark_cases.append(mvit_benchmark_case(
            BENCHMARK_ROOT, batch_size=BENCHMARK_BATCH_SIZE,
        ))
    else:
        benchmark_cases.extend(cnn_benchmark_cases(
            BENCHMARK_ROOT, model_names=[name], batch_size=BENCHMARK_BATCH_SIZE,
        ))
# To use a different experiment's weights, set case.checkpoint explicitly here.
# Do not infer a checkpoint's training dataset from the notebook's OUT variable.
for case in benchmark_cases:
    print(case.name, case.checkpoint, case.input_shape, case.state_policy)


In [ ]:
# Does not call either training routine. Each model runs in its own process.
benchmark_results = run_benchmarks(benchmark_cases, BENCHMARK_CONFIG, isolated=True)
for result in benchmark_results:
    print(result["name"], result["status"], result.get("weights_status"))
    if result.get("error"):
        print("  ", result["error"])
    unsupported = result.get("compute_details", {}).get("unsupported_operators", {})
    if unsupported:
        print("  MAC count unavailable for operators", sorted(unsupported))


In [ ]:
benchmark_table = results_to_dataframe(benchmark_results)
display(benchmark_table)

# Compact supplementary table. Keep missing/untrained/error labels visible.
paper_columns = [
    "Model", "Active params (M)", "Trainable params (M)",
    "GMAC/call (declared scope)", "Latency median [Q1,Q3] (ms/call)",
    "Peak allocated GPU (MiB)",
]
supplementary_table = benchmark_table[paper_columns].copy()
for index, result in enumerate(benchmark_results):
    if result["status"] != "ok" or result.get("weights_status") != "checkpoint_loaded":
        label = result["status"] if result["status"] != "ok" else result["weights_status"]
        supplementary_table.loc[index, "Model"] += f" [{label}]"
# Preserve differing input/state conditions when additional architectures are added.
if len({(r["mode"], r["state_policy"], r["batch_size"], r["frames_per_call"]) for r in benchmark_results}) > 1:
    supplementary_table["Workload"] = benchmark_table["Workload"]
for column in ["Comparison group", "Input specification"]:
    if column in benchmark_table:
        supplementary_table[column] = benchmark_table[column]
# MViTv2-S has the same source tensors but performs resizing inside its forward.
if any(r["case"]["metadata"].get("adapter_in_timed_forward") for r in benchmark_results):
    resolutions = []
    for result in benchmark_results:
        case = result["case"]
        source_size = case["metadata"].get("source_size")
        if source_size is None and case.get("input_factory") is None and len(case["input_shape"]) == 5:
            source_size = case["input_shape"][-2:]
        if source_size is None:
            resolutions.append("see input specification")
            continue
        network_size = case["metadata"].get("network_size", source_size)
        resolutions.append(f"{source_size[0]}x{source_size[1]} -> {network_size[0]}x{network_size[1]}")
    supplementary_table["Source -> network (pixels)"] = resolutions

display(supplementary_table)
print("Trainable params reflect requires_grad=True under each benchmark model configuration.")
print(MAC_SCOPE)
print("MViTv2-S latency/memory include its internal 112 -> 224 RGB input adapter.")
print("CPU runs have no GPU-memory measurement. Input frames/s in JSON is not end-to-end video throughput.")


In [ ]:
if SAVE_BENCHMARK_RESULTS:
    benchmark_paths = export_benchmark_report(benchmark_results, BENCHMARK_OUTPUT_DIR, table=supplementary_table)
    # The full report preserves statuses, checkpoint hashes, raw timings and metadata.
    print({name: str(path) for name, path in benchmark_paths.items()})
else:
    print("Set SAVE_BENCHMARK_RESULTS=True and rerun this cell to export CSV, JSON, Markdown and LaTeX.")


### Adding clip-based and streaming models later

Add a `BenchmarkCase` to `benchmark_cases`; the benchmark does not depend on the CNN training registry. Put model factories, custom input factories and state adapters in an **importable module under src/** (or an installed package), so separate worker processes can import them. A factory takes its configured kwargs and returns a CPU `torch.nn.Module`; loading and device transfer happen afterward. Models can have any number of output classes and tensor/dict/tuple outputs.

For a normal clip model, configure its required input shape/range/layout. If it needs normalized RGB, text tokens, several tensors or another input structure, provide `input_factory=Factory("my_models:make_inputs")`. That function receives `device`, floating `dtype`, and `seed`; it returns already prepared model inputs. The default adapter expands dicts into keyword arguments and tuples into positional arguments. Text token tensors should keep their integer dtype. Preprocessing is outside the forward timing.

For streaming models, provide an adapter factory receiving `model` and exposing `reset()` plus `step(inputs)`. Its reset must clear **all** internal/external cache. `state_policy="carry"` preserves bounded state between calls within each trial and measures the post-warm-up stream. `"reset_each_call"` measures independent starts. Each trial and measurement phase starts a new stream. Reset time is excluded. `frames_per_call` means **new frames per item**, not the cache window length.

Illustrative registration (replace the module names and paths with real implementations; this is not a bundled MoViNet implementation):

```python
benchmark_cases.append(BenchmarkCase(
    name="Streaming MoViNet",
    model_factory=Factory("my_models:build_movinet"),
    checkpoint=str(BENCHMARK_ROOT / "weights" / "movinet.pt"),
    input_shape=(1, 3, 1, 172, 172), input_range=(0.0, 1.0),
    mode="stream", state_policy="carry", frames_per_call=1,
    adapter_factory=Factory("my_models:MoViNetStateAdapter"),
    input_description="BCTHW; normalized RGB; one new frame",
    comparison_group="stream_1frame_172x172",
))
```

A custom adapter managing an explicit state argument can follow this interface:

```python
class MyStateAdapter:
    def __init__(self, model):
        self.model = model
        self.state = None

    def reset(self):
        self.state = None
        # Also call the actual model's internal-cache reset API, if it has one.

    def step(self, inputs):
        prediction, self.state = self.model(inputs, self.state)
        return prediction
```

The state API is implementation-specific. Adapt it to the chosen MoViNet package. Use `checkpoint_loader=Factory("my_models:load_weights")` for a different checkpoint format; it receives `model` and CPU checkpoint `path` and returns small JSON-compatible metadata.

Use `Factory("model_benchmark:count_forward_macs")` for a model whose forward operations are covered by the shared counter. The benchmark reports unclassified operators and leaves its total blank when any are encountered. For other architectures, supply a validated counter and parameter selector where available. A parameter selector receives `model` and returns active parameter names, including frozen weights. A custom MAC counter receives `model`, `adapter`, `inputs` and returns `{"macs_per_call": number, "scope": "operators covered"}`; document its coverage and convention. Neither callback edits the model.

Matching `comparison_group`, input resolution, new frames per call, batch size, dtype and state policy are needed for a matched-workload comparison. A one-frame streaming step and a 16-frame clip forward answer different latency questions. Retain input specifications and state conventions in the supplementary table/caption. These measurements do not produce accuracy or establish whole-pipeline real-time performance.
